# Part 3: Ranking & Filtering

#### Imports

In [1]:
import nltk
from collections import defaultdict
from array import array
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import math
import numpy as np
import collections
from numpy import linalg as la
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

#### Useful code from part 1 & part 2

In [2]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
def remove_punctuation(text):
    cleaned = ""
    for char in text:
        if char.isalnum() or char.isspace() or char == "-":
            cleaned += char
        else:
            cleaned += " "
    return cleaned


In [4]:
products_path = '../../data/fashion_products_dataset.json'
with open(products_path, "r", encoding="utf-8") as f:
    products = pd.read_json(products_path)

def build_terms(line):
    """
    Preprocess a line:
    ●  Removing stop words 
    ●  Tokenization 
    ●  Removing punctuation marks 
    ●  Stemming 
    ●  Transforming to lowercase

    Argument:
    line -- string (text) to be preprocessed

    Returns:
    line - a list of tokens corresponding to the input text after the preprocessing
    """

    stemmer = PorterStemmer()
    stop_words = set(stopwords.words("english"))
    line = line.lower()
    line = remove_punctuation(line)
    line = line.split()
    line = [x for x in line if x not in stop_words]
    line = [stemmer.stem(word) for word in line]
    return line

def get_products_information(products_df):
    elements = ["pid", "title", "description", "brand", "category", "sub_category", 
                "product_details", "seller", "out_of_stock", "selling_price", 
                "discount", "actual_price", "average_rating", "url"]
    
    products_df = products_df[elements]
    
    return products_df

products = get_products_information(products)
products["processed_title"] = products["title"].apply(build_terms)
products["processed_description"] = products["description"].apply(build_terms)
products['cat_subcat'] = products['category'] + ": " + products['sub_category']

In [5]:
products["title_description"] = products["processed_title"] + products["processed_description"]
display(products.head(5))

,pid,title,description,brand,category,sub_category,product_details,seller,out_of_stock,selling_price,discount,actual_price,average_rating,url,processed_title,processed_description,cat_subcat,title_description
0,TKPFCZ9EA7H5FYZH,Solid Women Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,York,Clothing and Accessories,Bottomwear,"[{'Style Code': '1005COMBO2'}, {'Closure': 'El...",Shyam Enterprises,False,921,69% off,"2,999",3.9,https://www.flipkart.com/yorker-solid-men-mult...,"[solid, women, multicolor, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",Clothing and Accessories: Bottomwear,"[solid, women, multicolor, track, pant, yorker..."
1,TKPFCZ9EJZV2UVRZ,Solid Men Blue Track Pants,Yorker trackpants made from 100% rich combed c...,York,Clothing and Accessories,Bottomwear,"[{'Style Code': '1005BLUE'}, {'Closure': 'Draw...",Shyam Enterprises,False,499,66% off,"1,499",3.9,https://www.flipkart.com/yorker-solid-men-blue...,"[solid, men, blue, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",Clothing and Accessories: Bottomwear,"[solid, men, blue, track, pant, yorker, trackp..."
2,TKPFCZ9EHFCY5Z4Y,Solid Men Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,York,Clothing and Accessories,Bottomwear,"[{'Style Code': '1005COMBO4'}, {'Closure': 'El...",Shyam Enterprises,False,931,68% off,"2,999",3.9,https://www.flipkart.com/yorker-solid-men-mult...,"[solid, men, multicolor, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",Clothing and Accessories: Bottomwear,"[solid, men, multicolor, track, pant, yorker, ..."
3,TKPFCZ9ESZZ7YWEF,Solid Women Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,York,Clothing and Accessories,Bottomwear,"[{'Style Code': '1005COMBO3'}, {'Closure': 'El...",Shyam Enterprises,False,911,69% off,"2,999",3.9,https://www.flipkart.com/yorker-solid-men-mult...,"[solid, women, multicolor, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",Clothing and Accessories: Bottomwear,"[solid, women, multicolor, track, pant, yorker..."
4,TKPFCZ9EVXKBSUD7,"Solid Women Brown, Grey Track Pants",Yorker trackpants made from 100% rich combed c...,York,Clothing and Accessories,Bottomwear,"[{'Style Code': '1005COMBO1'}, {'Closure': 'Dr...",Shyam Enterprises,False,943,68% off,"2,999",3.9,https://www.flipkart.com/yorker-solid-men-brow...,"[solid, women, brown, grey, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",Clothing and Accessories: Bottomwear,"[solid, women, brown, grey, track, pant, yorke..."


## Score

#### TF-IDF

In [6]:
def create_index_tfidf_products(products):
    """
    Build an inverted index and compute TF, DF, and IDF values for a product catalog.

    This function takes a pandas DataFrame of products where each row has:
      - 'pid': product ID
      - 'title_description': tokenized list of words from title + description
      - 'title': product title (optional)

    It returns:
      - index: dictionary mapping each term to postings lists 
               (product ID + positions where the term appears)
      - tf: normalized term frequency values for each term across products
      - df: document frequency (how many products contain each term)
      - idf: inverse document frequency (log-scaled rarity of each term)
      - title_index: mapping product IDs to their titles
    """

    index = defaultdict(list)       
    tf = defaultdict(list)         
    df = defaultdict(int)           
    idf = defaultdict(float)        # term -> inverse doc frequency
    title_index = defaultdict(str)  # pid -> product title

    num_products = len(products)

    # loop through each product
    for i in range(num_products):
        pid = products.iloc[i]["pid"]
        words = products.iloc[i]["title_description"]
        title_index[pid] = products.iloc[i].get("title", "")

        current_product_index = {}  # term -> [pid, positions]

        # record positions of each term in this product
        for position, term in enumerate(words):
            try:
                current_product_index[term][1].append(position)
            except:
                current_product_index[term] = [pid, array('I', [position])]

        # compute normalization factor for tf
        norm = math.sqrt(sum(len(posting[1]) ** 2 for posting in current_product_index.values()))

        # calculate tf and df
        for term, posting in current_product_index.items():
            tf[term].append(np.round(len(posting[1]) / norm, 4))  # normalized frequency
            df[term] += 1                                         # count product occurrence

        # add postings to main index
        for term, posting in current_product_index.items():
            index[term].append(posting)

    # calculate idf for each term
    for term in df:
        idf[term] = np.round(np.log(float(num_products / df[term])), 4)

    return index, tf, df, idf, title_index

In [7]:
def rank_products(terms, docs, index, idf, tf, title_index):
    """
    Perform the ranking of the results of a search based on the tf-idf weights

    Argument:
    terms -- list of query terms
    docs -- list of documents, to rank, matching the query
    index -- inverted index data structure
    idf -- inverted document frequencies
    tf -- term frequencies
    title_index -- mapping between page id and page title

    Returns:
    Print the list of ranked documents
    """

    
    # I'm interested only on the element of the docVector corresponding to the query terms
    # The remaining elements would became 0 when multiplied to the query_vector
    doc_vectors = defaultdict(lambda: [0] * len(terms)) # I call doc_vectors[k] for a nonexistent key k, the key-value pair (k,[0]*len(terms)) will be automatically added to the dictionary
    query_vector = [0] * len(terms)

    # compute the norm for the query tf
    query_terms_count = collections.Counter(terms)  # get the frequency of each term in the query.
    # Example: collections.Counter(["hello","hello","world"]) --> Counter({'hello': 2, 'world': 1})
    # HINT: use when computing tf for query_vector

    query_norm = la.norm(list(query_terms_count.values()))

    for termIndex, term in enumerate(terms):  #termIndex is the index of the term in the query
        if term not in index:
            continue

        ## Compute tf*idf(normalize TF as done with documents)
        query_vector[termIndex]= query_terms_count[term]/query_norm * idf[term] #query_vector[0] corresponds to the first term in the query

        # Generate doc_vectors for matching docs
        for doc_index, (doc, postings) in enumerate(index[term]):
            # Example of [doc_index, (doc, postings)]
            # 0 (26, array('I', [1, 4, 12, 15, 22, 28, 32, 43, 51, 68, 333, 337]))
            # 1 (33, array('I', [26, 33, 57, 71, 87, 104, 109]))
            # term is in doc 26 in positions 1,4, .....
            # term is in doc 33 in positions 26,33, .....

            #tf[term][0] will contain the tf of the term "term" in the doc 26
            if doc in docs: #if the odcument is in the list of documents retrieved (matching the query)
                doc_vectors[doc][termIndex] = tf[term][doc_index] * idf[term]  # TODO: check if multiply for idf

    # Calculate the score of each doc
    # compute the cosine similarity between queyVector and each docVector:
    # HINT: you can use the dot product because in case of normalized vectors it corresponds to the cosine similarity
    # see np.dot

    doc_scores=[[np.dot(curDocVec, query_vector), doc] for doc, curDocVec in doc_vectors.items() ]
    doc_scores.sort(reverse=True)
    result_docs = [x[1] for x in doc_scores]
    #print document titles instead if document id's
    #result_docs=[ title_index[x] for x in result_docs ]
    if len(result_docs) == 0:
        print("No results found, try again")
        query = input()
        docs = search_tf_idf(query, index)
    #print ('\n'.join(result_docs), '\n')
    return result_docs

def search_tf_idf(query, index, idf, tf, title_index):
    """
    Output is the list of documents that contain all of the query terms.
    So, we will get the list of documents for each query term, and take the intersection of them.
    """
    query = build_terms(query)

    docs = None
    
    for term in query:
        if term in index:
            # store in term_docs the ids of the docs that contain "term"
            term_docs= {posting[0] for posting in index[term]}

            if docs is None:
                docs = term_docs              
            else:
                docs &= term_docs             
        else:
            docs = set()
            break

    docs = list(docs)
    ranked_docs = rank_products(query, docs, index, idf, tf, title_index)
    return ranked_docs

In [8]:
index, tf, df, idf, title_index = create_index_tfidf_products(products)

print("Insert your query:\n")
query = input()
ranked_products = search_tf_idf(query, index, idf, tf, title_index)
top = 10

print("\n======================\nTop {} results out of {} for the query {}:\n".format(top, len(ranked_products), query))
for pid in ranked_products[:top]:
    url = products.loc[products["pid"] == pid, "url"].values
    url = url[0] if len(url) > 0 else "N/A"
    print("product_id = {} - product_title: {} - url: {}".format(pid, title_index[pid], url))

Insert your query:


Top 10 results out of 4189 for the query Men Round Neck:

product_id = TSHFUNN3QMCGSNCD - product_title: Printed Men Round Neck Pink T-Shirt - url: https://www.flipkart.com/steenbok-printed-men-round-neck-pink-t-shirt/p/itmae63cbd6b5047?pid=TSHFUNN3QMCGSNCD&lid=LSTTSHFUNN3QMCGSNCDGDM2OV&marketplace=FLIPKART&srno=b_6_207&otracker=browse&fm=organic&iid=7dd83f84-43c0-46f3-a29f-63779a8cefdc.TSHFUNN3QMCGSNCD.SEARCH&ssid=tdxf27zmww0000001612414998315
product_id = TSHFUNN2WF5PB3NZ - product_title: Printed Men Round Neck White T-Shirt - url: https://www.flipkart.com/steenbok-printed-men-round-neck-white-t-shirt/p/itmfe45ce2367365?pid=TSHFUNN2WF5PB3NZ&lid=LSTTSHFUNN2WF5PB3NZUVN2IE&marketplace=FLIPKART&srno=b_5_186&otracker=browse&fm=organic&iid=2af8fd4d-1e8c-4045-af31-31e8bf01b4d6.TSHFUNN2WF5PB3NZ.SEARCH&ssid=gp2ltka56o0000001612414324854
product_id = TSHFUNN2H8DUMJYQ - product_title: Printed Men Round Neck Blue T-Shirt - url: https://www.flipkart.com/steenbok-printed-men-r

#### BM25

In [9]:
def create_index_bm25_products(products):
    """
    Build an inverted index and compute BM25-related statistics for a product catalog.

    This function takes a pandas DataFrame of products where each row has:
      - 'pid': product ID
      - 'title_description': tokenized list of words from title + description
      - 'title': product title (optional)

    It returns:
      - index: dictionary mapping each term to postings lists 
               (product ID + positions where the term appears)
      - tf: raw term frequency values for each term across products
      - df: document frequency (how many products contain each term)
      - idf: inverse document frequency (BM25-style log-scaled rarity of each term)
      - title_index: mapping product IDs to their titles
      - doc_len: dictionary mapping product IDs to their document lengths
      - avg_doc_len: average document length across all products
    """

    index = defaultdict(list)          # term -> postings list [pid, positions]
    df = defaultdict(int)              # term -> number of documents containing term
    tf = defaultdict(list)             # term -> raw frequency counts across documents
    title_index = defaultdict(str)     # pid -> product title
    doc_len = {}                       # pid -> length of document (# of terms)
    N = len(products)                  # total number of products

    # loop through each product
    for i in range(N):
        pid = products.iloc[i]["pid"]
        words = products.iloc[i]["title_description"]
        title_index[pid] = products.iloc[i].get("title", "")
        doc_len[pid] = len(words)

        term_positions = {}  

        # record positions of each term in this product
        for pos, term in enumerate(words):
            if term in term_positions:
                term_positions[term].append(pos)
            else:
                term_positions[term] = [pos]

        # update index, tf, and df
        for term, positions in term_positions.items():
            index[term].append([pid, array('I', positions)])  # postings list
            df[term] += 1                                     # count product occurrence
            tf[term].append(len(positions))                   # raw frequency in this product

    # calculate idf for each term (BM25-style)
    idf = {}
    for term, freq in df.items():
        idf[term] = math.log(N / freq)

    # compute average document length
    avg_doc_len = sum(doc_len.values()) / N

    return index, tf, df, idf, title_index, doc_len, avg_doc_len

In [10]:
def rank_products_bm25(terms, docs, index, idf, tf, doc_len, avg_doc_len, k1, b):
    """
    Rank the results of a search using BM25 scoring.

    Arguments:
    terms -- list of query terms
    docs -- list of candidate documents (intersection of query terms)
    index -- inverted index data structure
    idf -- BM25 inverse document frequencies
    tf -- raw term frequencies
    doc_len -- dictionary mapping product IDs to document lengths
    avg_doc_len -- average document length across all products
    k1, b -- BM25 parameters

    Returns:
    result_docs -- list of ranked document IDs (sorted by BM25 score)
    """

    scores = defaultdict(float)  # doc_id -> BM25 score

    # loop through each query term
    for term_index, term in enumerate(terms):
        if term not in index:
            continue

        postings = index[term]       # list of [positions]
        tf_list = tf[term]           # raw term frequencies aligned with postings
        idf_value = idf[term]        # BM25 idf for this term

        # loop through postings for this term
        for i, (doc_id, positions) in enumerate(postings):
            if doc_id not in docs:
                continue

            f = tf_list[i]           # raw frequency of term in this doc
            dl = doc_len[doc_id]     # length of this document

            # BM25 scoring formula
            denom = f + k1 * (1 - b + b * dl / avg_doc_len)
            score = idf_value * ((f * (k1 + 1)) / denom)

            scores[doc_id] += score  # accumulate score for this doc

    # sort documents by score (highest first)
    doc_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    result_docs = [doc for doc, score in doc_scores]

    if len(result_docs) == 0:
        print("No results found, try again")
        query = input("Enter a new query: ")
        docs = search_bm25(query, index, idf, tf, {}, doc_len, avg_doc_len, k1, b)

    return result_docs


def search_bm25(query, index, idf, tf, title_index, doc_len, avg_doc_len, k1, b):
    """
    Perform a BM25 search over the product catalog.

    Arguments:
    query -- raw query string
    index -- inverted index data structure
    idf -- BM25 inverse document frequencies
    tf -- raw term frequencies
    title_index -- mapping between product ID and product title
    doc_len -- dictionary mapping product IDs to document lengths
    avg_doc_len -- average document length across all products
    k1, b -- BM25 parameters

    Returns:
    ranked_docs -- list of ranked document IDs (sorted by BM25 score)
    """

    terms = build_terms(query)  # tokenize query
    docs = None

    # collect documents containing all query terms (intersection)
    for term in terms:
        if term not in index:
            return []   # no results if any query term is missing

        term_docs = {posting[0] for posting in index[term]}
        if docs is None:
            docs = term_docs
        else:
            docs &= term_docs

        if not docs:
            return []   

    docs = list(docs)
    ranked_docs = rank_products_bm25(terms, docs, index, idf, tf, doc_len, avg_doc_len, k1, b)
    return ranked_docs

In [11]:
index, tf, df, idf, title_index, doc_len, avg_doc_len= create_index_bm25_products(products)
k1 = 1.5
b = 0.75

print("Insert your query:\n")
query = input()
ranked_products = search_bm25(query, index, idf, tf, title_index, doc_len, avg_doc_len, k1, b)
top = 10

print("\n======================\nTop {} results out of {} for the query {}:\n".format(top, len(ranked_products), query))
for pid in ranked_products[:top]:
    url = products.loc[products["pid"] == pid, "url"].values
    url = url[0] if len(url) > 0 else "N/A"
    print("product_id = {} - product_title: {} - url: {}".format(pid, title_index[pid], url))

Insert your query:


Top 10 results out of 4189 for the query Men Round Neck:

product_id = TSHFUNN2WF5PB3NZ - product_title: Printed Men Round Neck White T-Shirt - url: https://www.flipkart.com/steenbok-printed-men-round-neck-white-t-shirt/p/itmfe45ce2367365?pid=TSHFUNN2WF5PB3NZ&lid=LSTTSHFUNN2WF5PB3NZUVN2IE&marketplace=FLIPKART&srno=b_5_186&otracker=browse&fm=organic&iid=2af8fd4d-1e8c-4045-af31-31e8bf01b4d6.TSHFUNN2WF5PB3NZ.SEARCH&ssid=gp2ltka56o0000001612414324854
product_id = TSHFUNN2H8DUMJYQ - product_title: Printed Men Round Neck Blue T-Shirt - url: https://www.flipkart.com/steenbok-printed-men-round-neck-blue-t-shirt/p/itmaf8fc0a89c140?pid=TSHFUNN2H8DUMJYQ&lid=LSTTSHFUNN2H8DUMJYQLZYWJX&marketplace=FLIPKART&srno=b_5_195&otracker=browse&fm=organic&iid=2af8fd4d-1e8c-4045-af31-31e8bf01b4d6.TSHFUNN2H8DUMJYQ.SEARCH&ssid=gp2ltka56o0000001612414324854
product_id = TSHFUNN3QMCGSNCD - product_title: Printed Men Round Neck Pink T-Shirt - url: https://www.flipkart.com/steenbok-printed-men-r

#### Our own score system

In [12]:
def my_ranking(terms, docs, index, idf, tf, doc_len, title_index):
    """
    Rank the results of a search using harmonic mean of term scores.

    Formula:
      term_score = (tf / doc_len) * idf
    Final score = harmonic mean of all term scores for the query.

    Arguments:
    terms -- list of query terms
    docs -- list of candidate documents (intersection of query terms)
    index -- inverted index data structure (term -> postings [doc_id, positions])
    idf -- inverse document frequencies
    tf -- raw term frequencies aligned with postings order
    doc_len -- dictionary mapping product IDs to document lengths
    title_index -- mapping between product ID and product title

    Returns:
    result_docs -- list of ranked document IDs (sorted by harmonic mean score)
    """

    # store per-document scores
    scores = defaultdict(list)  # doc_id -> list of term scores

    # loop through each query term
    for term in terms:
        if term not in index:
            continue

        postings = index[term]       # list of [doc_id, positions]
        tf_list = tf[term]           # raw term frequencies aligned with postings
        idf_value = idf[term]        # idf for this term

        # loop through postings for this term
        for i, (doc_id, positions) in enumerate(postings):
            if doc_id not in docs:
                continue

            f = tf_list[i]                   # raw frequency of term in this doc
            tf_norm = f / doc_len[doc_id]    # normalize by document length
            term_score = tf_norm * idf_value # weighted score

            scores[doc_id].append(term_score)

    # compute harmonic mean for each document
    doc_scores = {}
    for doc_id, term_scores in scores.items():
        if term_scores:
            # harmonic mean: n / sum(1/x)
            score = len(term_scores) / sum(1.0 / s for s in term_scores if s > 0)
            doc_scores[doc_id] = score

    # sort documents by score (highest first)
    result_docs = [x[0] for x in sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)]

    if len(result_docs) == 0:
        print("No results found, try again")
        query = input("Enter a new query: ")
        docs = my_search(query, index, idf, tf, doc_len, title_index)

    return result_docs


def my_search(query, index, idf, tf, doc_len, title_index):
    """
    Perform a harmonic mean search over the product catalog.

    Arguments:
    query -- raw query string
    index -- inverted index data structure
    idf -- inverse document frequencies
    tf -- raw term frequencies aligned with postings order
    doc_len -- dictionary mapping product IDs to document lengths
    title_index -- mapping between product ID and product title

    Returns:
    ranked_docs -- list of ranked document IDs (sorted by harmonic mean score)
    """

    query_terms = build_terms(query)  # preprocess query into terms
    docs = None

    # collect documents containing all query terms (intersection)
    for term in query_terms:
        if term in index:
            term_docs = {posting[0] for posting in index[term]}
            if docs is None:
                docs = term_docs
            else:
                docs &= term_docs
        else:
            docs = set()
            break

    docs = list(docs)
    ranked_docs = my_ranking(query_terms, docs, index, idf, tf, doc_len, title_index)
    return ranked_docs

In [13]:
index, tf, df, idf, title_index, doc_len, avg_doc_len = create_index_bm25_products(products)

print("Insert your query:\n")
query = input()
ranked_products = my_search(query, index, idf, tf, doc_len, title_index)
top = 10

print("\n======================\nTop {} results out of {} for the query '{}':\n".format(top, len(ranked_products), query))
for pid in ranked_products[:top]:
    url = products.loc[products["pid"] == pid, "url"].values
    url = url[0] if len(url) > 0 else "N/A"
    print("product_id = {} - product_title: {} - url: {}".format(pid, title_index[pid], url))

Insert your query:


Top 10 results out of 4189 for the query 'Men Round Neck':

product_id = TSHFUNN2WF5PB3NZ - product_title: Printed Men Round Neck White T-Shirt - url: https://www.flipkart.com/steenbok-printed-men-round-neck-white-t-shirt/p/itmfe45ce2367365?pid=TSHFUNN2WF5PB3NZ&lid=LSTTSHFUNN2WF5PB3NZUVN2IE&marketplace=FLIPKART&srno=b_5_186&otracker=browse&fm=organic&iid=2af8fd4d-1e8c-4045-af31-31e8bf01b4d6.TSHFUNN2WF5PB3NZ.SEARCH&ssid=gp2ltka56o0000001612414324854
product_id = TSHFUNN2H8DUMJYQ - product_title: Printed Men Round Neck Blue T-Shirt - url: https://www.flipkart.com/steenbok-printed-men-round-neck-blue-t-shirt/p/itmaf8fc0a89c140?pid=TSHFUNN2H8DUMJYQ&lid=LSTTSHFUNN2H8DUMJYQLZYWJX&marketplace=FLIPKART&srno=b_5_195&otracker=browse&fm=organic&iid=2af8fd4d-1e8c-4045-af31-31e8bf01b4d6.TSHFUNN2H8DUMJYQ.SEARCH&ssid=gp2ltka56o0000001612414324854
product_id = TSHFUNN3QMCGSNCD - product_title: Printed Men Round Neck Pink T-Shirt - url: https://www.flipkart.com/steenbok-printed-men

## Word2Vec

In [14]:
def tokens_to_vec(tokens, model):
    """
    Construct the word vectors

    Arguments:
    tokens -- list of words in title and description of the product
    model -- the model to be used

    Returns:
    word_vectors -- the word vectors of the products
    """

    word_vectors = [model.wv[t] for t in tokens if t in model.wv] # creation of word vectors

    if not word_vectors:
        word_vectors = np.zeros(model.vector_size)

    word_vectors = np.mean(word_vectors, axis=0) # mean of the assigned values to construct the final word vector
    return word_vectors


def build_products_matrix(token_lists, model):
    """
    Build the products matrix to compute the scores

    Arguments:
    token_lists -- list of lists of tokens per product
    model -- the model to be used

    Returns:
    product_matrix -- the matrix of product tokens
    """

    product_matrix = np.array([tokens_to_vec(tokens, model) for tokens in token_lists]) # construction of the product matrix
    return product_matrix


def rank_products_cosine(query, product_vectors, df, model, top_k=20):
    """
    Create the ranking of the k most similar products with respect to the query by cosine similarity.

    Arguments:
    query -- query to search
    product_vectors -- product vectors to compute the similarity
    df -- dataframe of products 
    model -- model to be used
    top_k -- the number of products to show

    Returns:
    results -- list of ranked products (product id, title, score)
    """
    
    query_tokens = build_terms(query) # preprocess query
    query_vec = tokens_to_vec(query_tokens, model).reshape(1, -1)
    scores = cosine_similarity(query_vec, product_vectors)[0] # cosine similarity computation
    top_idx = np.argsort(scores)[::-1][:top_k] # ordered highest k scores

    # array with the info of the products
    results = []
    for i in top_idx:
        pid = df.iloc[i]["pid"]
        title = df.iloc[i]["title"]
        score = float(scores[i])
        results.append((pid, title, score))

    return results

In [15]:
title_descriptions = products["title_description"].tolist() # We make a list of the title/description tokens per product
model = Word2Vec(sentences=title_descriptions, vector_size=100, window=5, min_count=1, workers=4) # We train the Word2Vec model
product_vectors = build_products_matrix(title_descriptions, model)

# The queries from the part 2
queries = [
    "casual half sleeve polo shirt for men",
    "light blue jeans slim fit",
    "trousers chino casual men",
    "black sports shoes",
    "fancy t-shirt"
]

# We print the query, the score, the product id, and its title
for q in queries:
    print(f"\nQuery: {q}")
    results = rank_products_cosine(q, product_vectors, products, model, top_k=20)
    for rank, (pid, title, score) in enumerate(results, 1):
        print(f"{rank:2d}. Score={score:.4f} | PID={pid} | {title}")


Query: casual half sleeve polo shirt for men
 1. Score=0.9137 | PID=TSHFJFVB4UGTZR4Q | Solid Men Polo Neck Grey T-Shirt
 2. Score=0.9130 | PID=TSHFHF38G2NMSNPW | Printed Men Polo Neck Grey T-Shirt
 3. Score=0.9114 | PID=SHTFMHXVET3RR3Y8 | Men Regular Fit Solid Button Down Collar Casual Shirt  (Pack of 2)
 4. Score=0.9114 | PID=SHTFMHXVQJMFQ2MM | Men Regular Fit Solid Button Down Collar Casual Shirt  (Pack of 2)
 5. Score=0.9114 | PID=SHTFMHXVUBSXXGXK | Men Regular Fit Solid Button Down Collar Casual Shirt  (Pack of 2)
 6. Score=0.9114 | PID=SHTFMHXVPFGYHW8Y | Men Regular Fit Solid Button Down Collar Casual Shirt  (Pack of 2)
 7. Score=0.9114 | PID=SHTFMHXVGH9FXUKD | Men Regular Fit Solid Button Down Collar Casual Shirt  (Pack of 2)
 8. Score=0.9114 | PID=SHTFMHXV84URWXKH | Men Regular Fit Solid Button Down Collar Casual Shirt  (Pack of 2)
 9. Score=0.9097 | PID=SHTFMHXVBHPMNEUH | Women Regular Fit Solid Button Down Collar Casual Shirt  (Pack of 2)
10. Score=0.9097 | PID=SHTFMHXVDBG5HG